# Milestone 4 — LLM Explanations, RAG, Safe Text-to-SQL and Agent Workflow

This is a **design/demo submission** for Milestone 4. It deliberately uses mocked retrieval and a safe SQL allowlist rather than connecting to private systems or executing generated SQL. A complete submission should implement, test and document each component.


In [ ]:
from dataclasses import dataclass
import re

@dataclass
class RetrievedChunk:
    source: str
    text: str

# Mock policy corpus; replace with approved documents and a real retrieval index.
policy_chunks = [
    RetrievedChunk("credit_policy.md", "Risk explanations must cite evidence and should not use protected attributes."),
    RetrievedChunk("model_governance.md", "High-impact lending decisions require human review and monitoring.")
]
def retrieve(query, k=2):
    terms = set(re.findall(r"[a-z]+", query.lower()))
    ranked = sorted(policy_chunks, key=lambda c: len(terms & set(re.findall(r"[a-z]+", c.text.lower()))), reverse=True)
    return ranked[:k]

def grounded_explanation(query):
    hits = retrieve(query)
    return {"answer": "Use documented evidence, avoid protected attributes, and route high-impact decisions for human review.", "sources": [h.source for h in hits]}
print(grounded_explanation("How should we explain a lending risk result?"))

# Safe Text-to-SQL demo: only exact allowlisted read-only query templates are accepted.
ALLOWED = {"show default rate": "SELECT default_flag, COUNT(*) AS n FROM loan_records GROUP BY default_flag"}
def safe_sql(question):
    q = question.strip().lower()
    if q not in ALLOWED:
        raise ValueError("Question not supported by the SQL allowlist")
    sql = ALLOWED[q]
    if not sql.lstrip().lower().startswith("select") or ";" in sql.rstrip(";"):
        raise ValueError("Unsafe SQL rejected")
    return sql
print(safe_sql("show default rate"))

# Simple rule-based agent router placeholder.
def route(question):
    q = question.lower()
    if any(w in q for w in ["policy", "explain", "evidence"]): return "RAG_Agent"
    if any(w in q for w in ["count", "rate", "sql", "database"]): return "SQL_Agent"
    return "Human_Review"
print(route("Explain the policy evidence"))
print("Limitations: mock retrieval, no live LLM, no production SQL database, no empirical evaluation yet.")
